# 03 — Exploratory Data Analysis

Uncover behavioral patterns by emotion.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

features = pd.read_csv('../data/processed/final_features.csv')
participants = pd.read_csv('../data/raw/participants.csv', sep=';')
print(features.shape)

## Emotion distribution

In [ ]:
counts = features['emotionIndex'].value_counts()
print(counts)
counts.plot(kind='bar', color=['#888780','#D85A30','#BA7517','#185FA5','#1D9E75'])
plt.title('Samples per emotion class')
plt.tight_layout()
plt.show()

## Flight time by emotion

In [ ]:
emotion_order = ['A','H','N','C','S']
fig, axes = plt.subplots(1, 2, figsize=(12,4))

for ax, col, title in [(axes[0],'median_flight','Median flight time (ms)'),
                        (axes[1],'mean_dwell','Mean dwell time (ms)')]:
    means = features.groupby('emotionIndex')[col].mean().reindex(emotion_order)
    colors = ['#D85A30','#BA7517','#888780','#1D9E75','#185FA5']
    ax.bar(means.index, means.values, color=colors)
    ax.set_title(title)
    ax.set_ylabel('ms')

plt.tight_layout()
plt.show()

## Correlation heatmap

In [ ]:
feat_cols = ['mean_dwell','median_flight','cv_flight','mean_del_freq','mean_tot_time']
corr = features[feat_cols].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()

## ANOVA F-scores

In [ ]:
from scipy import stats
for col in feat_cols:
    groups = [features[features['emotionIndex']==e][col].dropna() for e in ['N','H','S','A','C']]
    groups = [g for g in groups if len(g) > 2]
    f, p = stats.f_oneway(*groups)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    print(f'{col:20s}: F={f:.2f}, p={p:.4f} {sig}')